In [40]:
import json
import os
import math
from uuid import uuid4

# ============================================================
# 1) Percorsi dei file originali
# ============================================================

artists_in = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\filled\artists_filled_with_feat.json"
tracks_in  = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\filled\albums.json"

# Cartella output
output_dir = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali"
os.makedirs(output_dir, exist_ok=True)

artists_out = os.path.join(output_dir, "artists_final.json")
tracks_out  = os.path.join(output_dir, "tracks_final.json")

# ============================================================
# 2) Caricamento file
# ============================================================

with open(artists_in, "r", encoding="utf-8") as f:
    artists = json.load(f)

with open(tracks_in, "r", encoding="utf-8") as f:
    tracks = json.load(f)

print(f"Caricati: {len(artists)} artists, {len(tracks)} tracks")
 

Caricati: 104 artists, 11166 tracks


In [41]:
# ============================================================
# Funzione per riconoscere valori sporchi
# ============================================================

def is_raw_missing(v):
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    if isinstance(v, str) and v.strip().lower() in ["", "none", "null", "nan", "undefined"]:
        return True
    return False

def normalize_missing(v):
    if is_raw_missing(v):
        return "NULL"
    return v

# ============================================================
# Pulizia dataset completo
# ============================================================

def clean_dataset(data):
    for row in data:
        for col, val in row.items():
            row[col] = normalize_missing(val)

clean_dataset(artists)
clean_dataset(tracks)

print("✔ Missing convertiti in 'NULL'.")


✔ Missing convertiti in 'NULL'.


In [42]:
# fix su type e rimozione active - end 
# ============================================================
# Fix robusto della colonna "type"
# ============================================================

for a in artists:
    if "type" not in a:
        a["type"] = "NULL"
    else:
        a["type"] = normalize_missing(a["type"])

# ============================================================
# Rimuovere colonna "active-end" se presente
# ============================================================

for a in artists:
    if "active-end" in a:
        del a["active-end"]

print("✔ Sistemata colonna 'type' e rimossa 'active-end'.")
 

✔ Sistemata colonna 'type' e rimossa 'active-end'.


In [43]:
# tipi misti 
# ============================================================
# Individua colonne con tipi misti
# ============================================================

from collections import defaultdict

def detect_mixed_types(data, name):
    print(f"\n=== ANALISI TIPI IN {name} ===")

    type_map = defaultdict(set)

    for row in data:
        for col, val in row.items():
            type_map[col].add(type(val).__name__)

    mixed = {col: types for col, types in type_map.items() if len(types) > 1}

    if mixed:
        print("⚠️ Colonne con tipi misti:")
        for col, types in mixed.items():
            print(f" - {col}: {types}")
    else:
        print("✔ Nessuna colonna con tipi misti!")

    return mixed

mixed_artists = detect_mixed_types(artists, "ARTISTS")
mixed_tracks  = detect_mixed_types(tracks, "TRACKS")
 


=== ANALISI TIPI IN ARTISTS ===
⚠️ Colonne con tipi misti:
 - latitude: {'str', 'float'}
 - longitude: {'str', 'float'}

=== ANALISI TIPI IN TRACKS ===
⚠️ Colonne con tipi misti:
 - year: {'str', 'float'}
 - month: {'str', 'float'}
 - day: {'str', 'float'}
 - n_sentences: {'str', 'float'}
 - n_tokens: {'str', 'float'}
 - char_per_tok: {'str', 'float'}
 - avg_token_per_clause: {'str', 'float'}
 - bpm: {'str', 'float'}
 - rolloff: {'str', 'float'}
 - flux: {'str', 'float'}
 - rms: {'str', 'float'}
 - flatness: {'str', 'float'}
 - spectral_complexity: {'str', 'float'}
 - pitch: {'str', 'float'}
 - loudness: {'str', 'float'}
 - disc_number: {'str', 'float'}
 - track_number: {'str', 'float'}
 - duration_ms: {'str', 'int', 'float'}
 - explicit: {'str', 'bool'}
 - popularity: {'str', 'float'}


In [44]:
import math

def to_float(val):
    """Converte in float oppure restituisce 'NULL'."""
    if val == "NULL":
        return "NULL"
    try:
        return float(val)
    except:
        return "NULL"

def to_int(val):
    """Converte in int se possibile, altrimenti 'NULL'."""
    if val == "NULL":
        return "NULL"
    try:
        return int(float(val))
    except:
        return "NULL"

def to_bool(val):
    """Converte valori in booleani."""
    if val == "NULL":
        return "NULL"
    if isinstance(val, bool):
        return val
    if isinstance(val, str):
        v = val.strip().lower()
        if v in ["true", "1", "yes"]:
            return True
        if v in ["false", "0", "no"]:
            return False
    return "NULL"


# =============================================
# FIX TIPI — ARTISTS
# =============================================

for a in artists:
    a["latitude"]  = to_float(a.get("latitude"))
    a["longitude"] = to_float(a.get("longitude"))


# =============================================
# FIX TIPI — TRACKS
# =============================================

for t in tracks:

    # --- INT ---
    t["year"]   = to_int(t.get("year"))
    t["month"]  = to_int(t.get("month"))
    t["day"]    = to_int(t.get("day"))

    t["disc_number"]  = to_int(t.get("disc_number"))
    t["track_number"] = to_int(t.get("track_number"))
    t["duration_ms"]  = to_int(t.get("duration_ms"))
    t["popularity"]   = to_int(t.get("popularity"))

    # --- FLOAT ---
    float_cols = [
        "n_sentences", "n_tokens", "char_per_tok", "avg_token_per_clause",
        "bpm", "rolloff", "flux", "rms", "flatness",
        "spectral_complexity", "pitch", "loudness"
    ]

    for col in float_cols:
        t[col] = to_float(t.get(col))

    # --- BOOL ---
    t["explicit"] = to_bool(t.get("explicit"))


print("✔ Tutti i tipi misti sono stati corretti.")
 

✔ Tutti i tipi misti sono stati corretti.


In [45]:
# check finale 
import math
from collections import defaultdict

# ============================================================
# CHECK 1 — Che TUTTI i missing siano 'NULL' (INCLUSA 'type')
# ============================================================

def check_missing_all_NULL(data, dataset_name):
    print(f"\n=== CHECK MISSING IN {dataset_name.upper()} ===")
    all_ok = True

    for col in data[0].keys():
        non_null_dirty = 0
        for row in data:
            v = row.get(col)

            # valore valido come NULL
            if v == "NULL":
                continue

            # valori NON ammessi
            if v is None:
                non_null_dirty += 1
            elif isinstance(v, float) and math.isnan(v):
                non_null_dirty += 1
            elif isinstance(v, str) and v.strip().lower() in ["", "nan", "none", "undefined"]:
                non_null_dirty += 1

        if non_null_dirty > 0:
            print(f"⚠️  Colonna {col}: {non_null_dirty} valori NON rimpiazzati da 'NULL'")
            all_ok = False

    if all_ok:
        print("✔ Tutti i missing sono correttamente 'NULL'.")


# ============================================================
# CHECK SPECIFICO SU 'type'
# ============================================================

print("\n=== CHECK SPECIFICO: colonna 'type' ===")
bad_type_values = [a["type"] for a in artists if a["type"] != "NULL" and not isinstance(a["type"], str)]

if bad_type_values:
    print("⚠️ Valori NON validi in 'type':", bad_type_values[:10])
else:
    print("✔ La colonna 'type' contiene SOLO valori 'NULL' o stringhe valide.")


# ============================================================
# CHECK 2 — Tipi misti nelle colonne
# ============================================================

def detect_mixed_types(data, dataset_name):
    print(f"\n=== CHECK TIPI MISTI IN {dataset_name.upper()} ===")

    col_types = defaultdict(set)

    for row in data:
        for col, val in row.items():
            if val == "NULL":
                col_types[col].add("NULL")
            else:
                col_types[col].add(type(val).__name__)

    mixed = {}
    for col, types in col_types.items():
        # 'NULL' conta come categoria a parte
        real_types = types - {"NULL"}

        if len(real_types) > 1:
            mixed[col] = real_types

    if not mixed:
        print("✔ Nessuna colonna ha tipi misti — struttura coerente.")
    else:
        print("⚠️ Colonne con tipi misti trovate:")
        for col, types in mixed.items():
            print(f" - {col}: {types}")


# ============================================================
# ESECUZIONE DEI CHECK FINALI
# ============================================================

check_missing_all_NULL(artists, "ARTISTS")
check_missing_all_NULL(tracks, "TRACKS")

detect_mixed_types(artists, "ARTISTS")
detect_mixed_types(tracks, "TRACKS")

print("\n=== CHECK COMPLETATO ===")
 
 


=== CHECK SPECIFICO: colonna 'type' ===
✔ La colonna 'type' contiene SOLO valori 'NULL' o stringhe valide.

=== CHECK MISSING IN ARTISTS ===
✔ Tutti i missing sono correttamente 'NULL'.

=== CHECK MISSING IN TRACKS ===
✔ Tutti i missing sono correttamente 'NULL'.

=== CHECK TIPI MISTI IN ARTISTS ===
✔ Nessuna colonna ha tipi misti — struttura coerente.

=== CHECK TIPI MISTI IN TRACKS ===
✔ Nessuna colonna ha tipi misti — struttura coerente.

=== CHECK COMPLETATO ===


In [46]:
import json
import os

# ============================================================
# Percorso output
# ============================================================

output_dir = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali"
os.makedirs(output_dir, exist_ok=True)

artists_out = os.path.join(output_dir, "artistsFinal.json")
tracks_out  = os.path.join(output_dir, "tracksFinal.json")

# ============================================================
# Salvataggio file
# ============================================================

with open(artists_out, "w", encoding="utf-8") as f:
    json.dump(artists, f, indent=4, ensure_ascii=False)

with open(tracks_out, "w", encoding="utf-8") as f:
    json.dump(tracks, f, indent=4, ensure_ascii=False)

print("✔ File puliti salvati in:")
print(" -", artists_out)
print(" -", tracks_out)


✔ File puliti salvati in:
 - C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\artistsFinal.json
 - C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\tracksFinal.json


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 152-153: truncated \UXXXXXXXX escape (1403326668.py, line 1)